# Lecture 5: Causal Mediation Analysis

In this lecture, we will introduce the concept of **Causal Mediation Analysis**, and learn how to use it to trace a circuit inside models.

### ✍ Learning goals

By the end of the lesson, we hope you feel familiar with the following concept.

* **Causal Mediation Analysis**: intervening on each model component to measure its effect on model output.

## 0️⃣ Setup

In [ ]:
try:
    import google.colab
    is_colab = True
except ImportError:
    is_colab = False

In [ ]:
from IPython.display import clear_output

!pip install nnsight

clear_output()

In [ ]:
import torch
import pandas as pd
import plotly.express as px
import plotly.io as pio
from nnsight import LanguageModel

model = LanguageModel("HuggingFaceTB/SmolLM2-360M")
prompt = 'A Transformer is' # sample prompt to laod the model
with model.generate(prompt) as tracer:
  output = model.generator.output.save()
model.to('cuda')
clear_output()

## 1️⃣ Understanding internals by probing activations

Let's consider a simple task of identifying the location of famous landmarks.

In [ ]:
prompt= "Eiffel Tower is in the country of"
with model.generate(prompt) as tracer:
  output = model.generator.output.save()
print(model.tokenizer.decode(output)[0])

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Eiffel Tower is in the country of France.

The Eiffel Tower is a famous landmark in Paris, France. It is a


In [ ]:
prompt= "Tokyo Tower is in the country of"
with model.generate(prompt) as tracer:
  output = model.generator.output.save()
print(model.tokenizer.decode(output)[0])

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Tokyo Tower is in the country of Japan.

The city of Tokyo is the capital of Japan.

The city of Tokyo


How does the model correctly answer these questions? <br>
Let's first look at internals for each prompt using Logit Lens.

In [ ]:
prompt= "Eiffel Tower is in the country of"

layer_range = list(range(10, len(model.model.layers))) # start from layer 10 to save time

decoded_layers = []
with model.trace(prompt):
    for layer_index in layer_range:
        # get layer activation
        layer_activation = model.model.layers[layer_index].output
        layer_activation = model.model.norm(layer_activation)
        # pass the layer through the final LM head to predict next token
        decoded_layer_logits = model.lm_head(layer_activation)
        # apply softmax to get a probability distribution over the next token
        decoded_layer = torch.nn.functional.softmax(decoded_layer_logits, dim=-1)
        decoded_layers.append(decoded_layer.save())

decoded_layers = torch.cat(decoded_layers) # (num_layers, num_tokens, vocab_size)

# find the maximum probability and corresponding tokens for each position
probabilities, decoded_tokens = decoded_layers.max(dim=-1)

# decode token ids
decoded_tokens = [
    [model.tokenizer.decode(t.item()) for t in layer_tokens]
    for layer_tokens in decoded_tokens
]

input_tokens = [t.replace('Ġ', '') for t in model.tokenizer.tokenize(prompt)]

In [ ]:
pio.renderers.default = "colab" if is_colab else "notebook_connected+plotly_mimetype+png"

fig = px.imshow(
    probabilities.detach().cpu().float().numpy(),
    x=[f"{i}) {t}" for i, t in enumerate(input_tokens)],
    y=layer_range,
    color_continuous_scale=px.colors.diverging.RdYlBu,
    color_continuous_midpoint=0.50,
    text_auto=True,
    labels=dict(x="Input Tokens", y="Layers", color="Probability")
)

fig.update_layout(
    title='Logit Lens Visualization: Eiffel Tower',
    xaxis_tickangle=0
)

fig.update_traces(text=decoded_tokens, texttemplate="%{text}")
fig.show()

In [ ]:
prompt= "Tokyo Tower is in the country of"

layer_range = list(range(10, len(model.model.layers))) # start from layer 10 to save time

decoded_layers = []
with model.trace(prompt):
    for layer_index in layer_range:
        # get layer activation
        layer_activation = model.model.layers[layer_index].output
        layer_activation = model.model.norm(layer_activation)
        # pass the layer through the final LM head to predict next token
        decoded_layer_logits = model.lm_head(layer_activation)
        # apply softmax to get a probability distribution over the next token
        decoded_layer = torch.nn.functional.softmax(decoded_layer_logits, dim=-1)
        decoded_layers.append(decoded_layer.save())

decoded_layers = torch.cat(decoded_layers) # (num_layers, num_tokens, vocab_size)

# find the maximum probability and corresponding tokens for each position
probabilities, decoded_tokens = decoded_layers.max(dim=-1)

# decode token ids
decoded_tokens = [
    [model.tokenizer.decode(t.item()) for t in layer_tokens]
    for layer_tokens in decoded_tokens
]

input_tokens = [t.replace('Ġ', '') for t in model.tokenizer.tokenize(prompt)]

In [ ]:
pio.renderers.default = "colab" if is_colab else "notebook_connected+plotly_mimetype+png"

fig = px.imshow(
    probabilities.detach().cpu().float().numpy(),
    x=[f"{i}) {t}" for i, t in enumerate(input_tokens)],
    y=layer_range,
    color_continuous_scale=px.colors.diverging.RdYlBu,
    color_continuous_midpoint=0.50,
    text_auto=True,
    labels=dict(x="Input Tokens", y="Layers", color="Probability")
)

fig.update_layout(
    title='Logit Lens Visualization: Tokyo Tower',
    xaxis_tickangle=0
)

fig.update_traces(text=decoded_tokens, texttemplate="%{text}")
fig.show()

We can see that the **country** representation emerges at around layer 26 in both cases. <br> This suggests that the layer representations in these layers are driving the correct model prediction. <br> In fact, we can confirm this hypothesis by performing difference in means steering, for example.

However, this doesn't really tell us about **how** these country representations are calculated inside the model. <br>
In other words, we want to draw a computaitonal graph showing how the difference in the prompt ("Eiffel" vs "Tokyo") gets transformed into difference in the prediction ("France" vs "Japan"). <br>
<b>Causal Mediation Analysis</b> helps us address this problem by quantifying the contribution of each model component on the output behavior.

## 2️⃣ Causal Mediation Analysis

Let's formally characterize this model behavior. We know that the model correctly predicts "France" instead of "Japan" as a continuation of the phrase "Eiffel Tower is in the country of". This can be descirbed as <br>
$$P(\text{France}|c_E) > P(\text{Japan}|c_E)$$
Here we used $c_{E}$ to denote the phrase "Eiffel Tower is in the country of".

We can quantify the model confidence by the difference in log probability (log odds).
$$\text{LogOdds}(\text{France},\text{Japan}|c_E) := \text{log}P(\text{France}|c_E) - \text{log}P(\text{Japan}|c_E)$$

Using this metric, the fact that the model correctly predicts "France" after "Eiffel Tower is in the country of" can be written as $$\text{LogOdds}(\text{France},\text{Japan}|c_E)>0$$

Similarly, the fact that the model correctly predicts "Japan" after "Tokyo Tower is in the country of" can be written as $$\text{LogOdds}(\text{France},\text{Japan}|c_T)<0$$
where $c_{T}$ refers to the phrase "Tokyo Tower is in the country of."

Our goal, then, is to understand how the model flips the sign of the log odds depending on the context ($c_E$ vs $c_T$).<br>
Specifically, we want to clarify how each model component is involved in this change in log odds.

To do this, we perform interchange intervention on the intermediate model representations and measure its effect on the log odds.

Let's say we want to see how much the last layer (32nd layer) representation at the last token ("of") influences the model prediction.

We quantify this effect as follows:
$$\text{LogOdds}(\text{France},\text{Japan}|c_E)-\text{LogOdds}(\text{France},\text{Japan}|c_E,h^{\text{last},32}_E←h^{\text{last},32}_T)$$

Here $h^{\text{last},32}_E$ refers to the 32nd layer representation at the last token in the 'Eiffel Tower' context ($c_E$), whereas $h^{\text{last},32}_T$ refers to the 32nd layer representation at the last token in the 'Tokyo Tower' context ($c_T$).

$\text{LogOdds}(\text{France},\text{Japan}|c_E,h^{\text{last},32}_E←h^{\text{last},32}_T)$ is the log odds after the model intervention.<br>
To calculate this, we start with the 'Eiffel Tower' context, but use the last layer representation of the last token from the 'Tokyo Tower' context ($h^{\text{last},\,32}_T$) and run the rest of the model to get the log probabilities.

In other words, this metric quantifies how much the model changes the prediction (measured by the log odds) based on the intervention.

We can repeat this for each layer and each token position to track how the model component in each layer at each token position influences the model prediction.

In [ ]:
base_prompt = "Eiffel Tower is in the country of"
source_prompt = "Tokyo Tower is in the country of"

In [ ]:
base_next_word = "France"
source_next_word = "Japan"

In [ ]:
base_output = model.tokenizer(" "+base_next_word.strip())["input_ids"][0] # includes a space
source_output = model.tokenizer(" "+source_next_word.strip())["input_ids"][0] # includes a space

In [ ]:
# calculate the original log odds
with model.generate(base_prompt) as tracer:
    original_logits = model.output.logits[:, -1, :].save()
original_logprobs = original_logits.log_softmax(dim=-1)
original_logodds = (original_logprobs[0, base_output] - original_logprobs[0, source_output]).item()

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


In [ ]:
# collect source activations
source_hidden_states = []
with torch.no_grad():
    with model.trace(source_prompt):
        # get hidden states of all layers in the network
        source_hidden_states.append([
            layer.output.save()
            for layer in model.model.layers
        ])
source_hidden_states = source_hidden_states[0]

In [ ]:
# intervene and collect intervention results
token_idx_list = [[0,1],[2],[3],[4],[5],[6],[7]] # concatenate the two tokens corresponding to Eiffel and Tokyo

causal_effects = []
# iterate through all the layers
for layer_idx in range(model.config.num_hidden_layers):
    causal_effect_per_layer = []
    # iterate through all tokens
    for token_idx in token_idx_list:
        with torch.no_grad():
            with model.trace(base_prompt) as tracer:
                # change the value of the base activation to the source value
                model.model.layers[layer_idx].output[:, token_idx, :] = \
                    source_hidden_states[layer_idx][:, token_idx, :]

                # get intervened output
                intervened_logits = model.output.logits[:, -1, :]
                intervened_logprobs = intervened_logits.log_softmax(dim=-1)
                # calcualte the log odds with intervention
                intervened_logodds = (intervened_logprobs[0, base_output] - intervened_logprobs[0, source_output]).item()
                effect = (original_logodds - intervened_logodds).save()

            causal_effect_per_layer.append(effect)
    causal_effects.append(causal_effect_per_layer)

In [ ]:
# get the labels for the plot
base_tokens = [model.tokenizer.decode([token]).strip() for token in model.tokenizer(base_prompt+' '+base_next_word).input_ids]
source_tokens = [model.tokenizer.decode([token]).strip() for token in model.tokenizer(source_prompt+' '+source_next_word).input_ids]
label_tokens = [''.join([base_tokens[i] for i in tokens])+'=>'+''.join([source_tokens[i] for i in tokens]) for tokens in token_idx_list]

In [ ]:
# visualize our results!
fig = px.imshow(
    causal_effects,
    x=label_tokens,
    y=list(range(model.config.num_hidden_layers)),
    template='simple_white',
    color_continuous_scale=[[0, '#FFFFFF'], [1, "#A59FD9"]]
)

fig.update_layout(
    xaxis_title='token',
    yaxis_title='layer',
    yaxis=dict(autorange='min')
)

fig

Let's interpret this result!<br>
First, we see that not all model components influence the model predictions; in fact, the ones that actually drive the model predictions are localized to specific token positions.<br>
Second, there is a specific layer (layer 26) where the 'information' (whether the prompt is 'Eiffel Tower' context or 'Tokyo Tower' context) moves from 'Eiffel/Tokyo' to the last token of the prompt.
This is consistent with the result from Logit Lens!

### ✏ **Exercise 1**

The experiment above tests how the model translates the difference in the monument entities (the Eiffel Tower vs Tokyo Tower) into the difference in the countries they are in (France vs Japan).

We may also be interested in is how the model knows which property of the monument entity it should focus on.<br>
For example, how does the model answer which <b>country</b> the Eiffel Tower is in, and not which <b>city</b>?<br>
How can we investigate this aspect of model behavior?

Use the same base prompt and change the source prompt to answer this question!

In [ ]:
base_prompt = "Eiffel Tower is in the country of"
source_prompt = "" # YOUR SOURCE PROMPT

In [ ]:
base_next_word = "France"
source_next_word = "" # NEXT TOKEN FOR SOURCE PROMPT

In [ ]:
base_output = model.tokenizer(" "+base_next_word.strip())["input_ids"][0] # includes a space
source_output = model.tokenizer(" "+source_next_word.strip())["input_ids"][0] # includes a space

In [ ]:
# calculate the original log odds

In [ ]:
# collect source activations

In [ ]:
# intervene and collect intervention results

In [ ]:
# get the labels for the plot

In [ ]:
# visualize our results!

## 3️⃣ Exploring other types of model components

In the previous section, we saw that we can track the information flow across layers by measuring how much swapping each layer between the two contexts changes the model prediction (log odds).

The result showed that layers before layer 26 encode the information at the token corresponding to the building, while the subsequent layers encode information at the last token of the prompt.

This means that the information is somehow transported from the building token to the last token at around layer 26.

Which model components inside each layer (e.g., query, key, value, attention output, etc.) are responsible for this information flow?

To investigate this question, we will intervene on the attention output and see how the intervention affects model prediciton.

In [ ]:
base_prompt = "Eiffel Tower is in the country of"
source_prompt = "Tokyo Tower is in the country of"

In [ ]:
base_next_word = "France"
source_next_word = "Japan"

In [ ]:
base_output = model.tokenizer(" "+base_next_word.strip())["input_ids"][0] # includes a space
source_output = model.tokenizer(" "+source_next_word.strip())["input_ids"][0] # includes a space

In [ ]:
# calculate the original log odds
with model.generate(base_prompt) as tracer:
    original_logits = model.output.logits[:, -1, :].save()
original_logprobs = original_logits.log_softmax(dim=-1)
original_logodds = (original_logprobs[0, base_output] - original_logprobs[0, source_output]).item()

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


In [ ]:
# collect source activations
source_hidden_states = []
with torch.no_grad():
    with model.trace(source_prompt):
        # index into attention head outputs
        source_hidden_states.append([
            layer.self_attn.o_proj.input.save()
            for layer in model.model.layers
        ])
source_hidden_states = source_hidden_states[0]

In [ ]:
# intervene and collect intervention results
attn_dim = model.config.hidden_size // model.config.num_attention_heads # dimension of each head

causal_effects = []
# iterate through layers (24-32) to save time
for layer_idx in range(24,32):
    causal_effect_per_layer = []
    # iterate through all tokens
    for head_index in range(model.config.num_attention_heads):
        with torch.no_grad():
            with model.trace(base_prompt) as tracer:
                # change the value of the base activation to the source value
                attention_value = model.model.layers[layer_idx].self_attn.o_proj.input
                # change value only at attention head index of the last token
                attention_value[:, -1, head_index * attn_dim:(head_index + 1) * attn_dim] = \
                source_hidden_states[layer_idx][:, -1, head_index * attn_dim:(head_index + 1) * attn_dim]
                # put attention head back in
                model.model.layers[layer_idx].self_attn.o_proj.input = attention_value

                # get intervened output
                intervened_logits = model.output.logits[:, -1, :]
                intervened_logprobs = intervened_logits.log_softmax(dim=-1)
                # calcualte the log odds with intervention
                intervened_logodds = (intervened_logprobs[0, base_output] - intervened_logprobs[0, source_output]).item()
                effect = (original_logodds - intervened_logodds).save()

            causal_effect_per_layer.append(effect)
    causal_effects.append(causal_effect_per_layer)

In [ ]:
# visualize our results!
fig = px.imshow(
    causal_effects,
    x=list(range(model.config.num_attention_heads)),
    y=list(range(24,32)),
    template='simple_white',
    color_continuous_scale=[[0, '#FFFFFF'], [1, "#A59FD9"]]
)

fig.update_layout(
    xaxis_title='attention head',
    yaxis_title='layer',
    yaxis=dict(autorange='min')
)

fig

Let's unpack this result!<br>
As expected from the result in the previous section, we observe a large effect in layer 26.<br>
Interestingly, we see this effect only for a specific head (head 10) and not others.<br>
This means attention head 10 in layer 26 transfers context information ($c_E$ vs $c_T$) from the building tokens to the last token in the prompt.<br>
Neat result!

### ✏ **Exercise 2**

Let's do the same analysis on the example we had in Exercise 1!

In [ ]:
base_prompt = "Eiffel Tower is in the country of"
source_prompt = "" # YOUR SOURCE PROMPT

In [ ]:
base_next_word = "France"
source_next_word = "" # NEXT TOKEN FOR SOURCE PROMPT

In [ ]:
base_output = model.tokenizer(" "+base_next_word.strip())["input_ids"][0] # includes a space
source_output = model.tokenizer(" "+source_next_word.strip())["input_ids"][0] # includes a space

In [ ]:
# calculate the original log odds

In [ ]:
# collect source activations

In [ ]:
# intervene and collect intervention results

In [ ]:
# visualize our results!

## 4️⃣ Tracking states

Let's try a slightly more complicated example.

In [ ]:
base_prompt = "Alice poured coffee into a cup. Then, she put it in the fridge. Now, the cup is"
source_prompt =  "Alice poured coffee into a cup. Then, she drank it in one go. Now, the cup is"

In [ ]:
with torch.no_grad():
    with model.generate(base_prompt) as tracer:
      output = model.generator.output.save()
print(model.tokenizer.decode(output)[0])

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Alice poured coffee into a cup. Then, she put it in the fridge. Now, the cup is cold.

The coffee is cold because it is in the fridge.

The coffee is


In [ ]:
with torch.no_grad():
    with model.generate(source_prompt) as tracer:
      output = model.generator.output.save()
print(model.tokenizer.decode(output)[0])

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Alice poured coffee into a cup. Then, she drank it in one go. Now, the cup is empty.

The first time, she poured coffee into a cup. The second time, she


Answering these prompts require tracking the state of the cup across sentences. <br>
Namely, we expect the cup is hot & full after Alice pours the coffee into it, while we expect it will be cold once it is in the fridge, or empty once she drinks it.<br>
How does the model track these different states of the cup?

We can investigate this using Causal Mediation Analysis!<br>
Here, to save time, we will intervene on multiple tokens at the same time.

In [ ]:
base_output = model.tokenizer(" cold")["input_ids"][0] # includes a space
source_output = model.tokenizer(" empty")["input_ids"][0] # includes a space

In [ ]:
# calculate the original log odds
with model.generate(base_prompt) as tracer:
    original_logits = model.output.logits[:, -1, :].save()
original_logprobs = original_logits.log_softmax(dim=-1)
original_logodds = (original_logprobs[0, base_output] - original_logprobs[0, source_output]).item()

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


In [ ]:
# collect source activations
source_hidden_states = []
with torch.no_grad():
    with model.trace(source_prompt):
        # get hidden states of all layers in the network
        source_hidden_states.append([
            layer.output.save()
            for layer in model.model.layers
        ])
source_hidden_states = source_hidden_states[0]

In [ ]:
# intervene and collect intervention results
token_idx_list = [[0,1,2,3,4,5,6],[7,8,9,10,11,12,13,14,15],[16,17],[18,19],[20]]

causal_effects = []
# iterate through all the layers
for layer_idx in range(model.config.num_hidden_layers):
    causal_effect_per_layer = []
    # iterate through all tokens
    for token_idx in token_idx_list:
        with torch.no_grad():
            with model.trace(base_prompt) as tracer:
                # change the value of the base activation to the source value
                model.model.layers[layer_idx].output[:, token_idx, :] = \
                    source_hidden_states[layer_idx][:, token_idx, :]

                # get intervened output
                intervened_logits = model.output.logits[:, -1, :]
                intervened_logprobs = intervened_logits.log_softmax(dim=-1)
                # calcualte the log odds with intervention
                intervened_logodds = (intervened_logprobs[0, base_output] - intervened_logprobs[0, source_output]).item()
                effect = (original_logodds - intervened_logodds).save()

            causal_effect_per_layer.append(effect)
    causal_effects.append(causal_effect_per_layer)

In [ ]:
# visualize our results!
fig = px.imshow(
    causal_effects,
    zmax=1,
    x=['Alice poured coffee into a cup.','Then, she put it in the fridge/drank it in one go.','Now,','the cup','is'],
    y=list(range(model.config.num_hidden_layers)),
    template='simple_white',
    color_continuous_scale=[[0, '#FFFFFF'], [1, "#A59FD9"]]
)

fig.update_layout(
    xaxis_title='token',
    yaxis_title='layer',
    yaxis=dict(autorange='min')
)

fig

Let's interpret this result!<br>
We observe the state of the cup (whether it is cold or empty) is represented at the tokens "the cup"!

In general, language models tend to aggregate contextual information at certain tokens (see [Li, Guo & Andreas, 2025](https://arxiv.org/abs/2503.02854), for example).

### ✏ **Exercise 3**

For this exercise, we will use workbench (https://workbench-git-v2-ndif.vercel.app/).

Let's consider the following two scenarios.

(1)
> Alice returned the book to Sarah because she finished it.<br><br>
> Question: Who does “she” refer to?<br><br>
> Answer:

(2)
> Alice returned the book to Sarah because she needed it.<br><br>
> Question: Who does “she” refer to?<br><br>
> Answer:

In (1), the one finishing the book is most likely Alice, so the answer should be Alice.<br>
In contrast, in (2), the one needing the book is most likely Sarah, so the answer should be Sarah.

This is an example of a Winograd Schema, a task that requires a situation model, a schema based on world knowledge.<br>
In the example above, in order to perform the pronoun disambiguation (i.e., knowing which noun the pronoun refers to), the model has to know different reasons a person may return a book.

Let's use workbench to see how the model solves this.

To make sure the model is performant enough to solve this task, use **llama-3.3-70B-Instruct**.

[1] Swap the last token to make sure the last token influences the model prediction.<br><br>
[2] Swap words in the middle of the sentence to see how that changes the model prediction. Which token seems to have a large effect? (Hint: it may take too much time to explore all possible token positions. At which token position would the model aggregate information?)<br><br>
[3] Change the names of the source sentence from Alice and Sarah to Bec and Cynthia, while keeping the base sentence the same. Swap the pronouns ('she') in the 'Question.' What kind of result do you expect? Did you get what you expected?

# 5️⃣ Multi-source Intervention

In this section, we will use multiple source sentences to get 'compositional' effects.

Let's go back to the Eiffel Tower example.<br>
Given the base prompt "Eiffel Tower is in the country of", we now want to not only change the monument entity (Eiffel Tower => Tokyo Tower), but also change which property of the monument entity the model should answer (the country => the city).

In other words, our goal is to give the model the base prompt ("Eiffel Tower is in the country of") but steer the model to answer "Tokyo" (not "Japan" like we did in section 2️⃣).

In section 3️⃣, we observed attention output at specific layers represent the country information ("France" vs "Japan") and in the exercise, we saw the same result for the the aspect information ("country" vs "city").

If this interpretation is correct, by combining the two interventions, we should be able to overwrite the country information and the property information and make the model say "Tokyo."

In [ ]:
# we now use two source prompts
base_prompt = "Eiffel Tower is in the country of"
source_prompt_1 = "Tokyo Tower is in the country of"
source_prompt_2 = "Eiffel Tower is in the city of"

In [ ]:
with model.generate(base_prompt) as tracer:
  output = model.generator.output.save()
print(model.tokenizer.decode(output)[0])

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Eiffel Tower is in the country of France.

The Eiffel Tower is a famous landmark in Paris, France. It is a


In [ ]:
with model.generate(source_prompt_1) as tracer:
  output = model.generator.output.save()
print(model.tokenizer.decode(output)[0])

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Tokyo Tower is in the country of Japan.

The city of Tokyo is the capital of Japan.

The city of Tokyo


In [ ]:
with model.generate(source_prompt_2) as tracer:
  output = model.generator.output.save()
print(model.tokenizer.decode(output)[0])

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Eiffel Tower is in the city of Paris, France.

The Eiffel Tower is a 300-meter (9


In [ ]:
base_output = model.tokenizer(" France")["input_ids"][0] # includes a space
source_output_1 = model.tokenizer(" Japan")["input_ids"][0] # includes a space
source_output_2 = model.tokenizer(" Paris")["input_ids"][0] # includes a space
source_output = model.tokenizer(" Tokyo")["input_ids"][0] # includes a space

In [ ]:
source_hidden_states_1 = []
with torch.no_grad():
    with model.trace(source_prompt_1):
        # index into attention head outputs
        source_hidden_states_1.append([
            layer.self_attn.o_proj.input.save()
            for layer in model.model.layers
        ])
source_hidden_states_1 = source_hidden_states_1[0]

In [ ]:
source_hidden_states_2 = []
with torch.no_grad():
    with model.trace(source_prompt_2):
        # index into attention head outputs
        source_hidden_states_2.append([
            layer.self_attn.o_proj.input.save()
            for layer in model.model.layers
        ])
source_hidden_states_2 = source_hidden_states_2[0]

In [ ]:
# intervene and collect intervention results
token_idx = -1
logprobs = []
for interv_type in ['no_intervention','source_country','source_property','combined']:
    if interv_type=='no_intervention':
        interv_country, interv_property = False, False
    if interv_type=='source_country':
        interv_country, interv_property = True, False
    if interv_type=='source_property':
        interv_country, interv_property = False, True
    if interv_type=='combined':
        interv_country, interv_property = True, True
    with torch.no_grad():
        with model.trace(base_prompt) as tracer:
            # change the value of the base activation to the source value
            for layer_idx in range(model.config.num_hidden_layers):
                if layer_idx in [24,25,26,27] and interv_country: # intervene at layer 24-27 for the country information
                    model.model.layers[layer_idx].self_attn.o_proj.input[:, token_idx, :] = \
                        source_hidden_states_1[layer_idx][:, token_idx, :]
                if layer_idx in [0,1,2,3,4,5] and interv_property: # intervene at layer 0-5 for the property information
                    model.model.layers[layer_idx].self_attn.o_proj.input[:, token_idx, :] = \
                        source_hidden_states_2[layer_idx][:, token_idx, :]

            # get intervened output
            intervened_logits = model.output.logits[:, -1, :]
            intervened_logprobs = intervened_logits.log_softmax(dim=-1).save()
    logprobs.extend([[interv_type, 'France', intervened_logprobs[0,base_output].item()],
                     [interv_type, 'Japan', intervened_logprobs[0,source_output_1].item()],
                     [interv_type, 'Paris', intervened_logprobs[0,source_output_2].item()],
                     [interv_type, 'Tokyo', intervened_logprobs[0,source_output].item()]])
df = pd.DataFrame(logprobs,columns=['intervention','next token','log probability'])

In [ ]:
fig = px.bar(df, x="intervention", y="log probability",
             color="next token",color_discrete_sequence=['#a6cee3','#b2df8a','#1f78b4','#33a02c'],
             barmode="group")
fig.show()

Let's unpack this result!
 - With no intervention, the most likely next token is "France", as expected.<br>
 - By intervening on layers 24-27 using representations from the "Tokyo Tower" prompt change the model prediction to "Japan." This is replication of section 3️⃣.<br>
 - By intervening on layers 0-5 using representations from the "city" prompt change the model prediction to "Paris." This is replication of exercise 2.<br>
 - Finally, by combining the two interventions, we manage to make the model predict "Tokyo"!

Why is this important?<br>
An alternative interpretation of the results from section 3️⃣ and exercise 2 is that these interventions are item-specific; they encode "Japan" information (not country information) or "Paris" information (not property information).<br>
Under this hypothesis, these interventions can force the model to predict "Japan" or "Paris", whatever the base prompt may be.

Our result here rejects this interpretation.<br>
Since "Tokyo" does not appear in either source prompts, the reason the model predicted "Tokyo" cannot be bacause we patched a "Tokyo" representation; instead this is a "compositional" effect emerging from two separate interventions!

## 🧠 Takeaways

In this lecture, we learned:
- Causal Mediation Analysis performs interchange intervention on each model component to measure its effect model output, which allows us to track context information across tokens and layers.
- Intervening on sub-layer components (such as attention outputs) reveals which attention heads are involved.
- Models sometimes store context information in intermediate token positions to track states in a long sequence.
- Combining multiple interventions can have compositional effects.

## 📖 References

Code is based on the NNsight tutorial ([Causal Mediation II](https://nnsight.net/tutorials/tutorials/causal_mediation_analysis/causal_mediation_analysis_ii/)).

Causal Mediation Analysis on different phenomena:
- gender bias: [Vig et al., 2020](https://arxiv.org/abs/2004.12265).
- subject verb agreement [Finlayson et al., 2021](https://arxiv.org/abs/2106.06087).
- indirect object identification: [Wang et al., 2022](https://arxiv.org/abs/2211.00593).
- situation model: [Yamakoshi et al., 2023](https://arxiv.org/abs/2306.03882).
- filler gap dependency: [Boguraev, Potts &  Mahowald, 2025](https://arxiv.org/abs/2505.16002).